# Project 3 — 5G-connected vehicle: results analysis

This notebook reads the results of the OMNeT++ simulations and draws all the charts with **Plotly**.

**How to use it**
1. Run `./run_all.sh` in the OMNeT++ shell first: it runs the simulations and writes the CSV files into `data/`.
2. Then run the cells of this notebook from top to bottom (*Run All*).
3. Every chart is shown here and also saved in `figures/` as `.html` (interactive) and `.png` (for the report and the slides).

The animated highway GIF is made by the separate script `make_highway_gif.py`.

**Contents**
0. Setup and helper functions
1. Calibration checks (why these parameters, enough replications?, warm-up)
2. Verification against queueing theory (chart 01)
3. Main results (charts 02–08)
4. Animated chart (GIF 09)

## 0. Setup

Requirements: `pip install plotly kaleido pandas numpy scipy imageio nbformat ipykernel`

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats
import imageio.v2 as imageio

pio.renderers.default = "plotly_mimetype+notebook_connected"   # charts shown inside the notebook

DATA = "data"          # CSV files written by run_all.sh
OUT = "figures"        # where the charts are saved
os.makedirs(OUT, exist_ok=True)

# model constants (the same values as in omnetpp.ini)
ALPHA = 1.28e-7      # S = ALPHA * d^2
ISD = 500.0          # distance between two base stations [m]
OFFSET = 20.0        # distance between the road and the base stations [m]
BACKHAUL = 0.001     # BS -> server delay [s]

SPEEDS = [10, 20, 30, 40]
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]   # one colour per line
SYMBOLS = ["circle", "square", "triangle-up", "diamond", "x"]
GREY = "#8a8983"
TOP = 2000           # [ms] unstable scenarios are drawn at this height (top of the delay charts)

# start the browser used by kaleido only once (makes saving PNGs much faster)
try:
    import kaleido
    kaleido.start_sync_server(silence_warnings=True)
except Exception:
    pass

### Reading the data

`opp_scavetool` writes one row **per value**. The function below keeps only:
- rows of type `scalar` → the results (e.g. `endToEndDelay:mean`)
- rows of type `itervar` → the factors of the run (V, H, T, ...)

and returns a table with **one row per run**. A run is marked `collapsed` when the transmitter is busy more than 98 % of the time (the queue never empties).

In [2]:
def load_runs(config):
    """Reads data/<config>.csv and returns a table with ONE ROW PER RUN."""
    raw = pd.read_csv(os.path.join(DATA, config + ".csv"), low_memory=False)

    # results: one column per scalar name
    scalars = raw[raw["type"] == "scalar"].copy()
    scalars["value"] = scalars["value"].astype(float)
    runs = scalars.pivot_table(index="run", columns="name", values="value")

    # factors: one column per iteration variable (V, H, T, cv, Tv)
    itervars = raw[raw["type"] == "itervar"]
    for name in itervars["attrname"].unique():
        rows = itervars[itervars["attrname"] == name].set_index("run")
        runs[name] = rows["attrvalue"].astype(float)

    # the transmitter is (almost) never idle -> the queue grows forever
    runs["collapsed"] = runs["utilization:timeavg"] > 0.98
    return runs.reset_index()

### Statistics

For every scenario we have 10 independent replications. The mean and the 95 % confidence interval are

$$\bar{x} \pm t_{0.975,\,n-1}\,\frac{s}{\sqrt{n}}$$

`summarize()` applies this to each group of runs with the same factors.

In [3]:
def confidence_interval(values):
    """Mean and half-width of the 95% confidence interval (Student t)."""
    values = np.array(values, dtype=float)
    n = len(values)
    mean = values.mean()
    half_width = stats.t.ppf(0.975, n - 1) * values.std(ddof=1) / np.sqrt(n)
    return mean, half_width


def summarize(runs, group_by, metrics=("endToEndDelay:mean",)):
    """One row per scenario: mean and CI of each metric + how many replications collapsed."""
    rows = []
    for key, group in runs.groupby(group_by):
        if not isinstance(key, tuple):
            key = (key,)
        row = dict(zip(group_by, key))
        row["n_collapsed"] = int(group["collapsed"].sum())
        for m in metrics:
            row[m], row[m + "_ci"] = confidence_interval(group[m])
        rows.append(row)
    return pd.DataFrame(rows)

### Plotting helpers

- `style()` gives every chart the same look, with a **title** and a **subtitle**.
- Log axes are labelled only at 1, 10, 100, 1000 ...
- `add_delay_line()` draws the mean delay with its 95 % CI. **Unstable** scenarios (a replication collapsed, or mean delay above 2 s) have no meaningful mean: they are drawn as **empty markers on the top edge** of the chart.

In [4]:
def style(fig, title, subtitle, height=600):
    fig.update_layout(title=dict(text=title, subtitle=dict(text=subtitle)),
                      template="simple_white", height=height, margin=dict(t=110),
                      font=dict(family="Segoe UI, Arial", size=14),
                      paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
                      legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center"))
    fig.update_xaxes(showgrid=True, gridcolor="#e8e7e3")
    fig.update_yaxes(showgrid=True, gridcolor="#e8e7e3")


def log_x(fig, title, row=None, col=None):
    fig.update_xaxes(type="log", dtick=1, title=title, row=row, col=col)


def log_delay_axis(fig, row=None, col=None):
    fig.update_yaxes(type="log", dtick=1, range=[0, np.log10(TOP * 1.5)],
                     title="mean E2E delay [ms] (log scale)", row=row, col=col)


def add_delay_line(fig, data, x, label, color, symbol="circle", dash="solid", row=None, col=None,
                   show_legend=True):
    data = data.sort_values(x)
    mean_ms = data["endToEndDelay:mean"] * 1000
    unstable = (data["n_collapsed"] > 0) | (mean_ms > TOP)
    stable = data[~unstable]
    collapsed = data[unstable]

    ci_ms = stable["endToEndDelay:mean_ci"] * 1000
    mean_ms = stable["endToEndDelay:mean"] * 1000
    fig.add_trace(go.Scatter(
        x=stable[x], y=mean_ms,
        # on a log axis an error bar cannot go below 0: the lower part is limited to 90% of the mean
        error_y=dict(array=ci_ms, arrayminus=np.minimum(ci_ms, 0.9 * mean_ms), width=4),
        mode="lines+markers", name=label, legendgroup=label, showlegend=show_legend,
        line=dict(color=color, width=2.2, dash=dash), marker=dict(color=color, size=8, symbol=symbol)),
        row=row, col=col)
    if len(collapsed) > 0:
        fig.add_trace(go.Scatter(
            x=collapsed[x], y=[TOP] * len(collapsed), mode="markers", name=label + " (collapsed)",
            legendgroup=label, showlegend=False,
            marker=dict(color="white", size=11, symbol=symbol, line=dict(width=2, color=color))),
            row=row, col=col)


def save(fig, name, width=1100, height=600):
    fig.write_html(os.path.join(OUT, name + ".html"), include_plotlyjs="cdn")
    fig.write_image(os.path.join(OUT, name + ".png"), width=width, height=height, scale=2)

### Load the two main scenarios

`summary` has one row per scenario (distribution × E[T] × V × H = 2 × 3 × 4 × 12 = 288 rows).

In [5]:
exp = load_runs("Exponential")
exp["dist"] = "Exponential"
logn = load_runs("Lognormal")
logn["dist"] = "Lognormal"

metrics = ["endToEndDelay:mean", "utilization:timeavg", "misassociated:mean", "txDistance:mean"]
summary = summarize(pd.concat([exp, logn]), ["dist", "T", "V", "H"], metrics)
summary.to_csv(os.path.join(OUT, "summary_table.csv"), index=False)
print(len(summary), "scenarios,", len(exp) + len(logn), "runs")
summary.head()

288 scenarios, 2880 runs


,dist,T,V,H,n_collapsed,endToEndDelay:mean,endToEndDelay:mean_ci,utilization:timeavg,utilization:timeavg_ci,misassociated:mean,misassociated:mean_ci,txDistance:mean,txDistance:mean_ci
0,Exponential,10.0,10.0,0.01,0,0.477831,0.007557,0.512097,0.000536,0.023622,0.000408,124.643486,0.113819
1,Exponential,10.0,10.0,0.02,0,0.050179,0.001892,0.405094,0.000781,0.005651,0.000262,127.643506,0.134337
2,Exponential,10.0,10.0,0.05,0,0.007488,0.000135,0.326109,0.000842,0.001462,0.000287,127.941977,0.140443
3,Exponential,10.0,10.0,0.10,0,0.006170,0.000069,0.298970,0.000846,0.001935,0.000543,127.947314,0.141169
4,Exponential,10.0,10.0,0.20,0,0.005809,0.000054,0.285405,0.000843,0.002853,0.001152,127.950886,0.141215


## 1. Calibration checks — why these parameters?

The project asks us to *"calibrate the scenarios so that meaningful results are obtained"*.
Calibrating does **not** mean finding the "best" numbers: it means choosing values that are **realistic** and that **make the effect we study visible**, without other effects hiding it.

*Sources:* project text `Project_03_5G_connected_vehicle.pdf` p. 1 ("it is up to the team to calibrate the scenarios so that meaningful results are obtained"); `2_Simulation.pdf` PDF p. 113 / slide 114 ("scenarios must be carefully engineered"); stability condition ρ < 1: `Notes on Queueing Theory` p. 47.

### 1a. The transmission constant α

With ideal association the farthest the car can be from its BS is half an ISD (250 m) plus the 20 m road offset, so the longest transmission is $S_{max} = \alpha\,(250^2 + 20^2)$ and the **local load at the cell edge** is $\rho_{edge} = S_{max} / E[T]$.

The table compares three choices of α:

In [6]:
rows = []
for alpha in [0.64e-7, 1.28e-7, 2.56e-7]:
    s_max = alpha * (250**2 + OFFSET**2)
    row = {"alpha [s/m^2]": alpha, "S at cell edge [ms]": round(s_max * 1000, 2)}
    for T in [10, 15, 20]:
        row[f"rho_edge, E[T]={T} ms"] = round(s_max / (T / 1000), 2)
    rows.append(row)
pd.DataFrame(rows)

,alpha [s/m^2],S at cell edge [ms],"rho_edge, E[T]=10 ms","rho_edge, E[T]=15 ms","rho_edge, E[T]=20 ms"
0,6.400000e-08,4.03,0.40,0.27,0.20
1,1.280000e-07,8.05,0.81,0.54,0.40
2,2.560000e-07,16.10,1.61,1.07,0.81


**How to read it**
- **α = 2.56·10⁻⁷** (double): ρ_edge = 1.61 > 1 with E[T] = 10 ms → the queue would explode at every cell edge **even with a perfect handover**. Our results would then show the radio being too slow, not the handover policy. ❌
- **α = 0.64·10⁻⁷** (half): ρ_edge = 0.40 → so little queueing that a wrong BS hardly matters; the handover effects become small. ❌
- **α = 1.28·10⁻⁷** (chosen): ρ_edge = 0.81 < 1 → stable with a perfect handover, but close enough to 1 that **a stale handover visibly hurts**. Every overload we see is caused by the handover procedure. ✅

The other parameters:

| Parameter | Value | Why |
|---|---|---|
| L, N | 2000 m, 5 BSs | ISD = 500 m, a realistic macro-cell spacing on a highway |
| V | 10, 20, 30, 40 m/s | 36–144 km/h, realistic highway speeds |
| E[T] | 10, 15, 20 ms | ρ_edge from 0.81 (loaded) down to 0.40 (light) |
| H | 0.01 … 20 s, log-spaced | wide enough to see **both** sides of the U-shaped curve (chart 02) |
| run length | 200 s warm-up + 2000 s | 2000 s = 5, 10, 15, 20 complete round trips for V = 10…40 m/s |
| replications | 10, `seed-set = ${repetition}` | checked in 1b |

### 1b. Are 10 replications enough?

The slides suggest a confidence interval of about **10 % of the mean**; if it is larger, add replications (up to ~30) — but *"when queueing systems are close to saturation ... there is little that you can do but acknowledge this fact"*.

*Source:* `2_Simulation.pdf` PDF p. 150 / slide 151. Replications and `seed-set = ${repetition}`: `3_OMNeT++.pdf` p. 54 and p. 86.

The chart shows the CI half-width as a percentage of the mean for every **stable** scenario.

In [7]:
stable = summary[(summary["n_collapsed"] == 0) & (summary["endToEndDelay:mean"] * 1000 < TOP)].copy()
stable["ci_percent"] = 100 * stable["endToEndDelay:mean_ci"] / stable["endToEndDelay:mean"]
print(f"stable scenarios: {len(stable)} of {len(summary)}")
print(f"CI / mean:  median {stable['ci_percent'].median():.1f} %,  "
      f"{(stable['ci_percent'] < 10).mean() * 100:.0f} % of the scenarios below 10 %")

fig = go.Figure()
for i, v in enumerate(SPEEDS):
    d = stable[(stable["dist"] == "Exponential") & (stable["T"] == 10) & (stable["V"] == v)].sort_values("H")
    fig.add_trace(go.Scatter(x=d["H"], y=d["ci_percent"], mode="lines+markers", name=f"V = {v} m/s",
                             line=dict(color=COLORS[i]), marker=dict(symbol=SYMBOLS[i], size=8)))
fig.add_trace(go.Scatter(x=[0.007, 30], y=[10, 10], mode="lines", name="10 % target",
                         line=dict(color="black", dash="dash", width=1.5)))
log_x(fig, "handover period H [s] (log scale)")
fig.update_yaxes(title="CI half-width / mean [%]", rangemode="tozero")
style(fig, "Precision of the results with 10 replications",
      "95% CI half-width as % of the mean delay (exponential T, E[T] = 10 ms, stable scenarios)", height=520)
fig.show()
save(fig, "00_ci_width", height=520)

stable scenarios: 254 of 288
CI / mean:  median 1.1 %,  84 % of the scenarios below 10 %


C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


**How to read it:** for every H up to 1 s the CI is **below ±6 %** of the mean (median over all stable scenarios: ±1.2 %). It goes above the 10 % target only for the last one or two values of H before the collapse (e.g. V = 20 m/s, H = 5 s).

We checked those scenarios with **30 replications** instead of 10:

| Scenario | 10 replications | 30 replications |
|---|---|---|
| V = 20 m/s, H = 5 s | 37.5 ms ± 75 % | 35.9 ms ± 37 % |
| V = 10 m/s, H = 10 s | 61.0 ms ± 81 % | 55.9 ms ± 41 % |

The CI shrinks as $1/\sqrt{n}$, the **mean does not change**, and the single replications range from 5.6 ms to 114 ms: close to the collapse the system is *sometimes fine, sometimes almost collapsed*, depending on where the car starts. This is the "close to saturation" case of the slides → we keep 10 replications and **report it** instead of hiding it.

### 1c. Warm-up period

The slides' method: run independent replications **without** warm-up, average them over time, and see when the average stops depending on the start (*not* "cut the first 10 %").

*Source:* `2_Simulation.pdf` PDF p. 123–124 / slides 124–125; `warmup-period` parameter: `3_OMNeT++.pdf` p. 56.
Here: config `TimeSeries` (3 replications, H = 1 s and 5 s, no warm-up removed), mean delay per 10 s window averaged over the replications.

In [8]:
raw_ts = pd.read_csv(os.path.join(DATA, "TimeSeries.csv"), low_memory=False)
H_of_run = raw_ts[(raw_ts["type"] == "itervar") & (raw_ts["attrname"] == "H")].set_index("run")["attrvalue"].astype(float)
vectors = raw_ts[(raw_ts["type"] == "vector") & (raw_ts["name"] == "endToEndDelay:vector")]

fig = go.Figure()
for i, h in enumerate([1.0, 5.0]):
    per_replica = []
    for run in vectors["run"]:
        if H_of_run[run] != h:
            continue
        row = vectors[vectors["run"] == run].iloc[0]
        t = np.array(row["vectime"].split(), dtype=float)
        delay = np.array(row["vecvalue"].split(), dtype=float)
        window = (t // 10).astype(int)                      # 10 s windows
        per_replica.append(pd.Series(delay).groupby(window).mean())
    average = pd.concat(per_replica, axis=1).mean(axis=1)    # average over the replications
    fig.add_trace(go.Scatter(x=average.index * 10, y=average.values * 1000, mode="lines",
                             name=f"H = {h:g} s (mean of 3 replications)", line=dict(color=COLORS[i], width=2)))
fig.add_trace(go.Scatter(x=[200, 200], y=[1, 1000], mode="lines", name="end of warm-up (200 s)",
                         line=dict(color="black", dash="dash", width=1.5)))
fig.update_yaxes(type="log", dtick=1, range=[0, 3.3], title="mean delay per 10 s window [ms]")
fig.update_xaxes(title="simulation time [s]")
style(fig, "Warm-up check", "Mean over replications of the delay per 10 s window, V = 30 m/s, no warm-up removed",
      height=520)
fig.show()
save(fig, "00_warmup", height=520)

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


**How to read it:** there is no initial trend — from the first cells the curves just repeat the same periodic pattern (a peak every time the car crosses a cell edge, every ISD/V ≈ 17 s). The transient lasts at most a few cells (~20–30 s), so **200 s of warm-up is a safe margin**.

**Conclusion of section 1:** the code needs no more fixes and no more calibration. The parameter choices are justified by the edge-load argument (1a), the precision check (1b) and the warm-up check (1c).

## 2. Verification against queueing theory — chart 01

Two cases where we know the exact answer:
*Source of the formula:* `Notes on Queueing Theory` p. 47 (Pollaczek–Khinchine, M/G/1).

- **(a) vehicle standing still** 125 m from a BS → S = 2 ms constant, exponential T → an **M/D/1** queue. Pollaczek–Khinchine: $E[R] = S + \frac{\rho S}{2(1-\rho)}$.
- **(b) moving vehicle, ideal association, no handover packets** → at each position u the queue is an M/D/1 with $S(u) = \alpha (u^2 + 20^2)$, and u is uniform in [0, 250 m] → we **average** the P-K formula over u.

In [9]:
md1 = summarize(load_runs("Validation_MD1"), ["Tv"]).sort_values("Tv", ascending=False)
S = ALPHA * 125 ** 2                                            # 2 ms
md1["rho"] = S / (md1["Tv"] / 1000)
md1["theory"] = S + md1["rho"] * S / (2 * (1 - md1["rho"]))    # P-K for M/D/1

ideal = summarize(load_runs("IdealHandover"), ["T", "V"])
u = np.linspace(0, ISD / 2, 10001)
S_u = ALPHA * (u ** 2 + OFFSET ** 2)
theory = {}
for T in ideal["T"].unique():
    rho_u = S_u / (T / 1000)
    theory[T] = np.mean(S_u + rho_u * S_u / (2 * (1 - rho_u))) + BACKHAUL
ideal["theory"] = ideal["T"].map(theory)

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "(a) vehicle standing still: M/D/1", "(b) ideal association: M/D/1 averaged over the position"])
fig.add_trace(go.Scatter(x=md1["rho"], y=md1["theory"] * 1000, mode="lines", name="theory",
                         line=dict(color=GREY, dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=md1["rho"], y=md1["endToEndDelay:mean"] * 1000, mode="markers",
                         name="OMNeT++ (mean, 95% CI)", marker=dict(size=11, color=COLORS[0]),
                         error_y=dict(array=md1["endToEndDelay:mean_ci"] * 1000)), row=1, col=1)
th = ideal.groupby("T")["theory"].first()
fig.add_trace(go.Scatter(x=th.index, y=th.values * 1000, mode="lines", showlegend=False,
                         line=dict(color=GREY, dash="dash")), row=1, col=2)
for i, v in enumerate(SPEEDS):
    d = ideal[ideal["V"] == v]
    fig.add_trace(go.Scatter(x=d["T"], y=d["endToEndDelay:mean"] * 1000, mode="markers", name=f"V = {v} m/s",
                             marker=dict(size=10, color=COLORS[i + 1], symbol=SYMBOLS[i]),
                             error_y=dict(array=d["endToEndDelay:mean_ci"] * 1000)), row=1, col=2)
fig.update_xaxes(title="utilisation ρ = S / E[T]", row=1, col=1)
fig.update_xaxes(title="E[T] [ms]", row=1, col=2)
fig.update_yaxes(title="mean E2E delay [ms]", rangemode="tozero")
style(fig, "Verification: the simulator reproduces queueing theory",
      "Dashed = Pollaczek-Khinchine formula, markers = OMNeT++ (10 replications)", height=540)
fig.show()
save(fig, "01_validation", height=540)

md1["error_%"] = 100 * (md1["endToEndDelay:mean"] / md1["theory"] - 1)
ideal["error_%"] = 100 * (ideal["endToEndDelay:mean"] / ideal["theory"] - 1)
print("max |error| M/D/1: %.2f %%,  ideal association: %.2f %%" % (md1["error_%"].abs().max(), ideal["error_%"].abs().max()))

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


max |error| M/D/1: 0.24 %,  ideal association: 0.36 %


**Takeaway:** the theory passes through all the simulation points — error ≤ 0.24 % for the M/D/1 and ≤ 0.4 % for the ideal-association case. In (b) the four speeds give the same delay: **with a perfect handover the speed does not matter** — any effect of V we see later comes from the handover procedure.

## 3. Main results

### Chart 02 — End-to-end delay vs. handover period H

**What it shows:** the mean delay of the application packets for every H (0.01 … 20 s) and every speed, with the ideal-association delay as a lower bound (dashed). Both axes are logarithmic.

In [10]:
data = summary[(summary["dist"] == "Exponential") & (summary["T"] == 10)]
lower_bound = ideal[ideal["T"] == 10]["endToEndDelay:mean"].mean() * 1000

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0.007, 30], y=[lower_bound, lower_bound], mode="lines",
                         name=f"ideal association (lower bound) = {lower_bound:.1f} ms",
                         line=dict(color="black", dash="dash", width=1.5)))
for i, v in enumerate(SPEEDS):
    add_delay_line(fig, data[data["V"] == v], "H", f"V = {v} m/s", COLORS[i], SYMBOLS[i])
log_x(fig, "handover period H [s] (log scale)")
log_delay_axis(fig)
style(fig, "End-to-end delay vs. handover period",
      "Exponential T, E[T] = 10 ms, mean ± 95% CI · empty markers on top = queue collapsed", height=640)
fig.show()
save(fig, "02_delay_vs_H", height=640)

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


**Takeaway — the delay is U-shaped in H.** Between H ≈ 0.1 s and 1 s the delay is 5.6–6.5 ms for every speed, only ~5 % above the lower bound. Too small H: up to 478 ms. Too large H: the queue collapses. *Why?* → chart 03.

### Chart 03 — Why the curve is U-shaped

**What it shows:** three quantities that explain the two sides of the U (stable scenarios only): the fraction of time the transmitter is busy, the % of application packets sent through a BS that is *not* the closest one, and the mean distance to the serving BS.

In [11]:
data = summary[(summary["dist"] == "Exponential") & (summary["T"] == 10) & (summary["n_collapsed"] == 0)]
panels = [("utilization:timeavg", 1, "transmitter utilisation"),
          ("misassociated:mean", 100, "packets via a non-closest BS [%]"),
          ("txDistance:mean", 1, "mean distance to serving BS [m]")]

fig = make_subplots(rows=1, cols=3, subplot_titles=[p[2] for p in panels], horizontal_spacing=0.08)
for i, v in enumerate(SPEEDS):
    d = data[data["V"] == v].sort_values("H")
    for c, (metric, factor, _) in enumerate(panels):
        fig.add_trace(go.Scatter(x=d["H"], y=d[metric] * factor, mode="lines+markers", name=f"V = {v} m/s",
                                 legendgroup=str(v), showlegend=(c == 0), line=dict(color=COLORS[i]),
                                 marker=dict(symbol=SYMBOLS[i], size=7)), row=1, col=c + 1)
log_x(fig, "H [s] (log scale)")
fig.update_yaxes(rangemode="tozero")
style(fig, "Why the curve is U-shaped",
      "Left side of the U: handover packets load the radio · right side: the car is served by a far BS", height=540)
fig.show()
save(fig, "03_mechanism", width=1400, height=540)

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


**Takeaway — two different causes.**
- **Left side (H too small):** handover packets steal transmitter time: utilisation goes from 0.28 to 0.51 at H = 0.01 s. Slower cars suffer more (478 ms at 10 m/s vs 130 ms at 40 m/s): near the cell edge app + handover packets give a local load ≈ 1.6 > 1, and a slow car stays longer in that zone.
- **Right side (H too large):** the car keeps using an old BS — up to 27 % of the packets at V = 40 m/s, H = 5 s — so the distance and S = α·d² grow.

### Chart 04 — The distance driven between two updates (V·H)

**What it shows:** the same results, but on the x-axis we put **V·H** = metres driven between two handover packets. The dotted line is ISD/2 = 250 m.

In [12]:
data = summary[(summary["dist"] == "Exponential") & (summary["T"] == 10)].copy()
data["VH"] = data["V"] * data["H"]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1, subplot_titles=[
    "misassociation depends only on V·H", "delay: V·H decides if the queue collapses"])
for i, v in enumerate(SPEEDS):
    d = data[data["V"] == v].sort_values("VH")
    st = d[d["n_collapsed"] == 0]
    fig.add_trace(go.Scatter(x=st["VH"], y=st["misassociated:mean"] * 100, mode="lines+markers",
                             name=f"V = {v} m/s", legendgroup=str(v), line=dict(color=COLORS[i]),
                             marker=dict(symbol=SYMBOLS[i], size=7)), row=1, col=1)
    add_delay_line(fig, d, "VH", f"V = {v} m/s", COLORS[i], SYMBOLS[i], row=1, col=2, show_legend=False)
fig.add_trace(go.Scatter(x=[ISD / 2, ISD / 2], y=[0, 17], mode="lines", showlegend=False,
                         line=dict(color=GREY, dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=[ISD / 2, ISD / 2], y=[1, TOP * 1.4], mode="lines", showlegend=False,
                         line=dict(color=GREY, dash="dot")), row=1, col=2)
log_x(fig, "V·H = metres driven between two handover packets [m]")
fig.update_yaxes(title="packets via a non-closest BS [%]", rangemode="tozero", row=1, col=1)
log_delay_axis(fig, row=1, col=2)
style(fig, "V·H is what matters on the right side of the U",
      "Exponential T, E[T] = 10 ms · dotted line: ISD/2 = 250 m", height=580)
fig.show()
save(fig, "04_delay_vs_VH", width=1300, height=580)

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


**Takeaway:** plotted against V·H the curves of the four speeds **lie on top of each other**. The queue collapses when V·H goes beyond ≈ 150–200 m, at every speed. Practical rule: choose **H ≈ (a few tens of metres) / V**.

### Chart 05 — Summary heatmap

**What it shows:** the mean delay [ms] for every (V, H) pair; top = exponential T, bottom = lognormal T with the same mean and variance.

**How to read it:** darker blue = longer delay (log colour scale). **Grey ✕** = the queue collapsed, so there is no meaningful mean. The **outlined, bold** cell in each row is the best H for that speed.

In [13]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.16,
                    subplot_titles=["Exponential T", "Lognormal T (same mean and variance as the exponential)"])

for r, dist in enumerate(["Exponential", "Lognormal"]):
    d = summary[(summary["dist"] == dist) & (summary["T"] == 10)]
    H_values = sorted(d["H"].unique())
    delay_z, collapsed_z, text, collapsed_text = [], [], [], []      # one row per speed

    for v in SPEEDS:
        delay_row, collapsed_row, text_row, collapsed_text_row = [], [], [], []
        for h in H_values:
            cell = d[(d["V"] == v) & (d["H"] == h)].iloc[0]
            ms = cell["endToEndDelay:mean"] * 1000
            if cell["n_collapsed"] > 0 or ms > TOP:          # unstable: grey cell with a cross
                delay_row.append(None)
                collapsed_row.append(1)
                text_row.append("")
                collapsed_text_row.append("✕")
            else:                                            # stable: coloured by the delay
                delay_row.append(np.log10(ms))
                collapsed_row.append(None)
                text_row.append(f"{ms:.1f}" if ms < 10 else f"{ms:.0f}")
                collapsed_text_row.append("")
        # best H for this speed = smallest stable delay -> bold + outline
        best = int(np.nanargmin([np.nan if z is None else z for z in delay_row]))
        text_row[best] = "<b>" + text_row[best] + "</b>"
        fig.add_shape(type="rect", x0=best - 0.5, x1=best + 0.5, y0=SPEEDS.index(v) - 0.5, y1=SPEEDS.index(v) + 0.5,
                      line=dict(color="black", width=3), fillcolor="rgba(0,0,0,0)", layer="above",
                      row=r + 1, col=1)
        delay_z.append(delay_row)
        collapsed_z.append(collapsed_row)
        text.append(text_row)
        collapsed_text.append(collapsed_text_row)

    x_labels = [f"{h:g}" for h in H_values]
    y_labels = [f"{v} m/s" for v in SPEEDS]
    ticks_ms = [5, 10, 20, 50, 100, 200, 500, 1000]
    fig.add_trace(go.Heatmap(z=delay_z, x=x_labels, y=y_labels, text=text, texttemplate="%{text}",
                             textfont=dict(size=13), colorscale="Blues", zmin=np.log10(4), zmax=np.log10(TOP),
                             xgap=3, ygap=3, showscale=(r == 0), hoverinfo="skip",
                             colorbar=dict(title="mean delay [ms]", tickvals=np.log10(ticks_ms),
                                           ticktext=[str(t) for t in ticks_ms], len=1.05, y=0.5)),
                  row=r + 1, col=1)
    fig.add_trace(go.Heatmap(z=collapsed_z, x=x_labels, y=y_labels, text=collapsed_text, texttemplate="%{text}",
                             textfont=dict(size=15, color="white"), colorscale=[[0, "#9a9993"], [1, "#9a9993"]],
                             xgap=3, ygap=3, showscale=False, hoverinfo="skip"),
                  row=r + 1, col=1)

fig.update_xaxes(title="handover period H [s]", type="category", showgrid=False)
fig.update_yaxes(title="speed V", type="category", showgrid=False)
style(fig, "Mean E2E delay [ms] for every speed V and handover period H",
      "E[T] = 10 ms · mean of 10 replications · grey ✕ = queue collapsed · outlined = best H for each speed",
      height=820)
fig.update_layout(showlegend=False)
fig.show()
save(fig, "05_heatmap_V_H", width=1400, height=820)

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


**Takeaway:** a "safe valley" of H between ~0.05 and 1–3 s, where the delay stays at 5–7 ms. The best H is 0.5–1 s for every speed. The collapsed region (grey, right) starts earlier for faster cars: at H = 5 s for 40 m/s, at H = 20 s for 10 m/s. The two distributions give almost the same map.

### Chart 06 — Distribution and variability of T

**What it shows:** delay vs H at V = 30 m/s for T with the **same mean (10 ms)** but different distribution / variability: exponential (CoV = 1) and lognormal with CoV = 0.5, 1, 2, 3.

In [14]:
e = exp[(exp["V"] == 30) & (exp["T"] == 10)].copy()
e["label"] = "exponential (CoV = 1)"
l = logn[(logn["V"] == 30) & (logn["T"] == 10)].copy()
l["label"] = "lognormal, CoV = 1"
cv = load_runs("LognormalCV")
cv["label"] = "lognormal, CoV = " + cv["cv"].map("{:g}".format)
dist_summary = summarize(pd.concat([e, l, cv]), ["label", "H"])
dist_summary.to_csv(os.path.join(OUT, "summary_distribution.csv"), index=False)

lines = [("lognormal, CoV = 0.5", COLORS[2], "solid"), ("exponential (CoV = 1)", COLORS[0], "solid"),
         ("lognormal, CoV = 1", COLORS[0], "dot"), ("lognormal, CoV = 2", COLORS[3], "solid"),
         ("lognormal, CoV = 3", COLORS[1], "solid")]
fig = go.Figure()
for i, (label, color, dash) in enumerate(lines):
    add_delay_line(fig, dist_summary[dist_summary["label"] == label], "H", label, color, SYMBOLS[i], dash=dash)
log_x(fig, "handover period H [s] (log scale)")
log_delay_axis(fig)
style(fig, "Same mean of T, different variability",
      "V = 30 m/s, E[T] = 10 ms · CoV = coefficient of variation of T", height=640)
fig.show()
save(fig, "06_distribution", height=640)

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


**Takeaway — the variability of T matters more than the distribution family** (as predicted by *"queueing arises from variability"*, `Notes on Queueing Theory` p. 48). With the same mean *and* variance (CoV = 1), the lognormal gives ~9 % *less* delay than the exponential. Changing the variability matters much more: at H = 0.5 s, CoV 0.5 → 4.0 ms, CoV 3 → 13.7 ms; with CoV = 3 the system is already unstable at H = 5 s.

### Chart 07 — Traffic load

**What it shows:** delay vs H at V = 30 m/s for three mean inter-arrival times E[T] = 10, 15, 20 ms (exponential T).

In [15]:
data = summary[(summary["dist"] == "Exponential") & (summary["V"] == 30)]
fig = go.Figure()
for i, T in enumerate([10, 15, 20]):
    add_delay_line(fig, data[data["T"] == T], "H", f"E[T] = {T} ms", COLORS[i], SYMBOLS[i])
log_x(fig, "handover period H [s] (log scale)")
log_delay_axis(fig)
style(fig, "Effect of the traffic load", "Exponential T, V = 30 m/s · smaller E[T] = more traffic", height=620)
fig.show()
save(fig, "07_load", height=620)

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


**Takeaway:** more traffic does not move the best H (≈ 0.2–1 s), but **narrows the range of safe H**: with E[T] = 10 ms the queue collapses at H = 10 s, with E[T] = 20 ms it still works (130 ms).

### Chart 08 — Delay over time

**What it shows:** the delay over time (mean per 5 s) for a stable (H = 1 s), a borderline (H = 5 s) and a collapsing (H = 20 s) case, 3 independent replications each (never conclude from a single trajectory).

In [16]:
H_list = [1.0, 5.0, 20.0]
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=[f"H = {h:g} s  (V·H = {30 * h:g} m)" for h in H_list])
for r, h in enumerate(H_list):
    runs_with_h = [run for run in vectors["run"] if H_of_run[run] == h]
    for k, run in enumerate(runs_with_h):
        row = vectors[vectors["run"] == run].iloc[0]
        t = np.array(row["vectime"].split(), dtype=float)
        delay = np.array(row["vecvalue"].split(), dtype=float)
        window = (t // 5).astype(int)                          # mean per 5 s
        mean_per_window = pd.Series(delay).groupby(window).mean()
        fig.add_trace(go.Scatter(x=mean_per_window.index * 5, y=mean_per_window.values * 1000, mode="lines",
                                 name=f"replica {k + 1}", legendgroup=str(k), showlegend=(r == 0),
                                 line=dict(color=COLORS[r], dash=["solid", "dash", "dot"][k], width=1.6)),
                      row=r + 1, col=1)
    fig.update_yaxes(type="log", dtick=1, range=[0, 5.5], title="delay [ms]", row=r + 1, col=1)
fig.update_xaxes(title="simulation time [s]", row=3, col=1)
style(fig, "End-to-end delay over time", "V = 30 m/s, E[T] = 10 ms · mean per 5 s · 3 replications per H",
      height=820)
fig.show()
save(fig, "08_timeseries", height=820)

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


**Takeaway — three regimes.** H = 1 s: small peaks at every cell edge. H = 5 s: peaks of ~1 s, the queue builds up and drains every cell. H = 20 s: the queue never drains and the delay settles at ~110 s (positive-feedback loop: handover packet stuck in the queue → wrong BS → larger S → longer queue).

## 4. Animated chart — GIF 09

One frame per speed: the U-curve of chart 02 drawn for V = 10, 20, 30, 40 m/s (previous speeds in grey). Saved as `figures/09_animated_speed.gif` (put it directly in PowerPoint).

In [17]:
data = summary[(summary["dist"] == "Exponential") & (summary["T"] == 10)]
frames = []
for i, v in enumerate(SPEEDS):
    fig = go.Figure()
    for j in range(i):                                    # previous speeds in light grey
        add_delay_line(fig, data[data["V"] == SPEEDS[j]], "H", f"V = {SPEEDS[j]} m/s", "#d6d5cf", SYMBOLS[j])
    add_delay_line(fig, data[data["V"] == v], "H", f"V = {v} m/s", COLORS[i], SYMBOLS[i])
    log_x(fig, "handover period H [s] (log scale)")
    log_delay_axis(fig)
    style(fig, f"How the U-curve changes with the speed — V = {v} m/s", "E[T] = 10 ms, exponential T", height=560)
    fig.update_xaxes(range=[-2.2, 1.5])
    frames.append(imageio.imread(fig.to_image(format="png", width=1000, height=560)))
imageio.mimsave(os.path.join(OUT, "09_animated_speed.gif"), frames, duration=1.3, loop=0)
fig.show()   # last frame
print("saved figures/09_animated_speed.gif")

C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


C:\Users\marli\AppData\Local\Temp\claude\35e6d4fbed761557\d5ede581-a94f-4c4d-bcd0-1452efb3e09e\scratchpad\venv2\Lib\site-packages\plotly\io\_kaleido.py:185: UserWarning: The kopts argument is ignored if using a server.
  img_bytes = kaleido.calc_fig_sync(


saved figures/09_animated_speed.gif
